# Traffic Situation Classifier (model-ready dataset)
Uses `traffic_model_ready.csv`, which already contains the numeric features, a numeric target and a date-based `Split` column.
Upload that file next to this notebook (Colab: left sidebar → Files → Upload).

In [ ]:
# ==========================================================
# CELL 1: Import Libraries
# ==========================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

In [ ]:
# ==========================================================
# CELL 2: Load the Dataset
# ==========================================================
df = pd.read_csv("traffic_model_ready.csv")

print(df.shape)
df.head()

## Data Preprocessing

In [ ]:
# ==========================================================
# CELL 3: Data Quality Check
# ==========================================================
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

# Total must equal the sum of the four vehicle counts
counts = ["CarCount", "BikeCount", "BusCount", "TruckCount"]
print("Total != sum of counts:", (df[counts].sum(axis=1) != df["Total"]).sum())

print("\nClass balance:")
print(df["Traffic_Situation"].value_counts(normalize=True).round(3))

print("\nRows per split:")
print(df["Split"].value_counts())

df[counts + ["Total"]].describe()

## Exploratory Data Analysis (EDA)

In [ ]:
# ==========================================================
# CELL 4: EDA
# ==========================================================
sns.set_theme(style="whitegrid")
order = ["low", "medium", "high"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Class distribution
sns.countplot(x="Traffic_Situation", hue="Traffic_Situation", data=df, order=order, palette="Set2", legend=False, ax=axes[0])
axes[0].set_title("Traffic Situation Distribution")

# Total vehicles per class
sns.boxplot(x="Traffic_Situation", y="Total", hue="Traffic_Situation", data=df, order=order, palette="Set2", legend=False, ax=axes[1])
axes[1].set_title("Total Vehicles vs Traffic Situation")

# Share of each class by hour of the day
by_hour = pd.crosstab(df["Hour"], df["Traffic_Situation"], normalize="index")[order]
by_hour.plot(kind="bar", stacked=True, ax=axes[2], color=["#66c2a5", "#fc8d62", "#8da0cb"], width=0.9)
axes[2].set_title("Traffic Situation by Hour of Day")
axes[2].set_xlabel("Hour")
axes[2].set_ylabel("Share of readings")

plt.tight_layout()
plt.show()

# Correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df[counts + ["Total", "Time_Minutes"]].corr(), annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

## Feature Engineering, Split and Scaling

In [ ]:
# ==========================================================
# CELL 5: Features, Target, Split and Scaling
# ==========================================================

# Feature engineering (same idea as before): share of heavy vehicles
df["Heavy_Vehicle_Ratio"] = (df["BusCount"] + df["TruckCount"]) / (df["Total"] + 1e-5)

# These are exactly the inputs the Streamlit app can supply
FEATURES = [
    "CarCount", "BikeCount", "BusCount", "TruckCount", "Total",
    "Heavy_Vehicle_Ratio",
    "Day_Num", "Is_Weekend",
    "Time_sin", "Time_cos",
]

# Target: 0 = low, 1 = medium, 2 = high
CLASS_NAMES = ["low", "medium", "high"]
TARGET = "Traffic_Label"

# Split by DATE using the Split column (not random), so neighbouring
# 15-minute readings never end up in both train and test
train = df[df["Split"] == "train"]
test = df[df["Split"] == "test"]

x_train, y_train = train[FEATURES], train[TARGET]
x_test, y_test = test[FEATURES], test[TARGET]

print("Train size:", len(x_train))
print("Test size :", len(x_test))

# Scale features (fit on train only)
scaler = StandardScaler()
X_train = scaler.fit_transform(x_train)
X_test = scaler.transform(x_test)

**Why these changes from the original notebook**
- Counts are numeric, so they are used as numbers. One-hot encoding `CarCount`, `BikeCount`, etc. turns every distinct count into its own column and throws away the fact that 40 is more than 30.
- `Date` is dropped (it is only a day-of-month ID with no month or year), and `Time` is already converted to `Time_sin` / `Time_cos`.
- This is a classification problem, so classifiers are used instead of regressors, and no label rounding or clipping is needed.

In [ ]:
# ==========================================================
# CELL 6: Logistic Regression
# ==========================================================
lr_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)
print("Logistic Regression trained successfully!")

In [ ]:
# ==========================================================
# CELL 7: Decision Tree
# ==========================================================
# max_depth keeps the tree from memorising the training data
dt_model = DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42)
dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)
dt_proba = dt_model.predict_proba(X_test)
print("Decision Tree trained successfully!")

In [ ]:
# ==========================================================
# CELL 8: Random Forest
# ==========================================================
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=12, class_weight="balanced",
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)
print("Random Forest trained successfully!")

## Model Evaluation

In [ ]:
# ==========================================================
# CELL 9: Model Evaluation & Comparison
# ==========================================================
def evaluate_model(name, y_true, y_pred, y_proba):
    return [
        name,
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, average="weighted"),
        recall_score(y_true, y_pred, average="weighted"),
        f1_score(y_true, y_pred, average="weighted"),
        f1_score(y_true, y_pred, average="macro"),
        roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted"),
    ]

results = pd.DataFrame(
    [
        evaluate_model("Logistic Regression", y_test, lr_pred, lr_proba),
        evaluate_model("Decision Tree", y_test, dt_pred, dt_proba),
        evaluate_model("Random Forest", y_test, rf_pred, rf_proba),
    ],
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 (weighted)", "F1 (macro)", "AUC"],
)
results.iloc[:, 1:] = results.iloc[:, 1:].round(4)

print("=" * 70)
print("MODEL PERFORMANCE COMPARISON (held-out dates)")
print("=" * 70)
display(results)

best = results.loc[results["F1 (macro)"].idxmax()]
print(f"\nBest model by macro F1: {best['Model']}  (accuracy {best['Accuracy']}, macro F1 {best['F1 (macro)']})")

results.set_index("Model")[["Accuracy", "F1 (macro)"]].plot(kind="bar", figsize=(8, 5), rot=15)
plt.title("Model Comparison")
plt.ylim(0, 1.05)
plt.show()

In [ ]:
# ==========================================================
# CELL 10: Confusion Matrices and Feature Importance
# ==========================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, pred) in zip(axes, [("Logistic Regression", lr_pred),
                                    ("Decision Tree", dt_pred),
                                    ("Random Forest", rf_pred)]):
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

print(classification_report(y_test, rf_pred, target_names=CLASS_NAMES))

# Which inputs does the Random Forest rely on?
imp = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values()
imp.plot(kind="barh", figsize=(7, 5), title="Random Forest Feature Importance")
plt.show()

**Reading these results:** the label is largely derived from the vehicle counts, so high scores here mostly show that the models recovered that rule, not that they can forecast future traffic. Time-only models (without the counts) score far lower.

## Save Files for the Streamlit App

In [ ]:
# ==========================================================
# CELL 11: File Generation
# ==========================================================
joblib.dump(lr_model, "logistic_model.pkl")
joblib.dump(rf_model, "random_forest_model.pkl")
joblib.dump(dt_model, "decision_tree_model.pkl")
joblib.dump(scaler, "scaler.pkl")

# The app needs the exact feature order and class names
joblib.dump({"features": FEATURES, "class_names": CLASS_NAMES}, "model_meta.pkl")

print("Saved: logistic_model.pkl, random_forest_model.pkl, decision_tree_model.pkl, scaler.pkl, model_meta.pkl")

In [ ]:
# ==========================================================
# CELL 12: Prediction function (matches the Streamlit UI inputs)
# ==========================================================
DAY_MAP = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3,
           "Friday": 4, "Saturday": 5, "Sunday": 6}

def build_input_row(car, bike, bus, truck, day_name, time_str):
    """Turn the UI inputs into one row with the same features used in training."""
    t = pd.to_datetime(time_str.strip(), format="%I:%M:%S %p")
    minutes = t.hour * 60 + t.minute
    total = car + bike + bus + truck
    day_num = DAY_MAP[day_name]
    row = {
        "CarCount": car, "BikeCount": bike, "BusCount": bus, "TruckCount": truck,
        "Total": total,
        "Heavy_Vehicle_Ratio": (bus + truck) / (total + 1e-5),
        "Day_Num": day_num,
        "Is_Weekend": int(day_num >= 5),
        "Time_sin": np.sin(2 * np.pi * minutes / 1440),
        "Time_cos": np.cos(2 * np.pi * minutes / 1440),
    }
    return pd.DataFrame([row])[FEATURES]

def predict_traffic(model, car, bike, bus, truck, day_name, time_str):
    X = scaler.transform(build_input_row(car, bike, bus, truck, day_name, time_str))
    proba = model.predict_proba(X)[0]
    label = CLASS_NAMES[int(np.argmax(proba))]
    return label, {c: round(float(p), 3) for c, p in zip(CLASS_NAMES, proba)}

# Example using the values shown in the app screenshot
for name, m in [("Logistic Regression", lr_model), ("Decision Tree", dt_model), ("Random Forest", rf_model)]:
    label, probs = predict_traffic(m, car=25, bike=5, bus=2, truck=10, day_name="Monday", time_str="12:00:00 AM")
    print(f"{name:20s} -> {label:7s} {probs}")